# Build Satellite Database (Flight 01)

End-to-end demo of **Step 5a** of the paper *Structure-Based UAV Visual Geo-Localization via Delaunay Triangulation and Ekeland Free-Cone Angle Descriptors*: build an offline KD-Tree index over a satellite GeoTIFF.

Each 500×500 satellite patch (stride 100 px) is processed by the SAME `segment_batch` function that the UAV image goes through in notebook 2 (Mask R-CNN ResNet-50-FPN trained at 500×500), so segmentation quality is identical on both branches. Then:

1. Mask R-CNN building segmentation
2. Contour → Douglas-Peucker polygon
3. Constrained Delaunay Triangulation
4. MFCA / Ekeland angle per triangle vertex
5. Assemble 5-D descriptor $f = (\alpha_1, \alpha_2, e_1, e_2, e_3)$
6. Tag with `patch_id`, persist into `SatelliteDatabase` (`scipy.spatial.KDTree(p=1)`).

Demo target: **`satellite01.tif`** (9774 × 26762, Changjiang-20).

## 1. Clone repo (Colab / Kaggle)

Skip this cell if you are already running the notebook from a cloned checkout.

In [1]:
import os

REPO_URL = 'https://github.com/kagtgi/LocalizationUAV.git'
REPO_DIR = 'LocalizationUAV'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pwd

d:\bk_study_stuff\paper6\LocalizationUAV\notebooks\LocalizationUAV


'pwd' is not recognized as an internal or external command,
operable program or batch file.


## 2. Install Python dependencies

In [2]:
!pip install --quiet -r requirements.txt

ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python >=3.8,<3.12; 1.10.0rc1 Requires-Python >=3.8,<3.12; 1.10.0rc2 Requires-Python >=3.8,<3.12; 1.10.1 Requires-Python >=3.8,<3.12; 1.11.0 Requires-Python >=3.9,<3.13; 1.11.0rc1 Requires-Python >=3.9,<3.13; 1.11.0rc2 Requires-Python >=3.9,<3.13; 1.11.1 Requires-Python >=3.9,<3.13; 1.11.2 Requires-Python >=3.9,<3.13; 1.11.3 Requires-Python >=3.9,<3.13; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python >=3.9,<3.13; 1.26.1 Requires-Python >=3.9,<3.13; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.7.2 Requires-Python >=3.7,<3.11; 1.7.3 Requires-Python >=3.7,<3.11; 1.8.0 Requires-Python >=3.8,<3.11; 1.8.0rc1 Requires-Python >=3.8,<3.11; 1.8.0rc2 Requi

## 3. Setup Kaggle API and download data

Upload your `kaggle.json` into the current working directory first (Colab: drag-and-drop into the file panel; Kaggle: not needed — datasets are already available).

In [3]:
import os

# Idempotent: only download + extract if the flight-01 data is not already present.
if not os.path.exists('UAV-VisLoc/01/satellite01.tif'):
    # Configure the Kaggle API (Colab: drag kaggle.json into the file panel first).
    !mkdir /.kaggle
    !mv kaggle.json /.kaggle
    !mv /.kaggle /root/
    !chmod 600 ~/.kaggle/kaggle.json

    !kaggle datasets download building-segment
    !kaggle datasets download hailong1610/uav-visloc-dataset
    !unzip -q building-segment.zip
    !unzip -q uav-visloc-dataset.zip
    print('UAV-VisLoc dataset downloaded and extracted (~17.7 GB, 11 flights).')
else:
    print('UAV-VisLoc/01 already present; skipping Kaggle download.')

The syntax of the command is incorrect.
'mv' is not recognized as an internal or external command,
operable program or batch file.
'mv' is not recognized as an internal or external command,
operable program or batch file.
'chmod' is not recognized as an internal or external command,
operable program or batch file.
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


Dataset URL: https://www.kaggle.com/datasets/hailong1610/uav-visloc-dataset
License(s): apache-2.0
Resuming from 15956180992 bytes (1763098288 bytes left)...

Connection error: OSError: [Errno 28] No space left on device
Retrying in 2.5 seconds... (attempt 1/5)
Retry 1/5: Resuming from 15956180992 bytes (1763098288 bytes left)...

Connection error: OSError: [Errno 28] No space left on device
Retrying in 4.2 seconds... (attempt 2/5)
Retry 2/5: Resuming from 15956180992 bytes (1763098288 bytes left)...

Connection error: OSError: [Errno 28] No space left on device
Retrying in 9.0 seconds... (attempt 3/5)
Retry 3/5: Resuming from 15956180992 bytes (1763098288 bytes left)...

Connection error: OSError: [Errno 28] No space left on device
Retrying in 16.9 seconds... (attempt 4/5)
Retry 4/5: Resuming from 15956180992 bytes (1763098288 bytes left)...

Connection error: OSError: [Errno 28] No space left on device
Retrying in 32.6 seconds... (attempt 5/5)
Retry 5/5: Resuming from 15956180992 byt


 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]
 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]

 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]
 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]

 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]
 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]

 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]
 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]

 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]
 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]

 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]
 90%|█████████ | 14.9G/16.5G [00:00<?, ?B/s]
[Errno 28] No space left on device
'unzip' is not recognized as an internal or external command,
operable program or batch file.


UAV-VisLoc dataset downloaded and extracted (~17.7 GB, 11 flights).


'unzip' is not recognized as an internal or external command,
operable program or batch file.


## 3b. Download the Mask R-CNN checkpoint

`best_model.pth` (~170 MB) is too large to live in git, so it is hosted on Google Drive:

https://drive.google.com/file/d/1jlRfOXYU18DcOjWEwNBet1b7FHCIClcB/view

The cell below uses [`gdown`](https://github.com/wkentaro/gdown) to fetch the file once and skips the download on subsequent runs.

In [4]:
import os

MODEL_GDRIVE_ID = '1jlRfOXYU18DcOjWEwNBet1b7FHCIClcB'
MODEL_LOCAL_PATH = 'best_model.pth'

if not os.path.exists(MODEL_LOCAL_PATH):
    !pip install --quiet gdown
    !gdown 'https://drive.google.com/uc?id={MODEL_GDRIVE_ID}' -O {MODEL_LOCAL_PATH}
else:
    print(f'{MODEL_LOCAL_PATH} already present; skipping download.')

print('best_model.pth size:', os.path.getsize(MODEL_LOCAL_PATH) // (1024 * 1024), 'MB')

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id='https://drive.google.com/uc?id=1jlRfOXYU18DcOjWEwNBet1b7FHCIClcB'

but Gdown can't. Please check connections and permissions.


FileNotFoundError: [WinError 2] The system cannot find the file specified: 'best_model.pth'

## 4. Imports & config

Define paths and hyperparameters (paper §4.6: 500-px patches, 100-px stride, KD-Tree leaf 40, ℓ₁). Then verify the satellite GeoTIFF and the Mask R-CNN checkpoint are present before the (slow) build.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / 'UAV-VisLoc'
FLIGHT_ID = '01'
MODEL_PATH = REPO_ROOT / 'best_model.pth'

PATCH_SIZE = 500
STRIDE = 100
SCORE_THRESHOLD = 0.5
INFERENCE_BATCH_SIZE = 4  # number of 500x500 patches per Mask R-CNN forward pass

OUTPUT_DIR = REPO_ROOT / 'outputs' / FLIGHT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_NPZ_PATH = OUTPUT_DIR / f'satellite{FLIGHT_ID}_kdtree.npz'
DESCRIPTORS_CSV_PATH = OUTPUT_DIR / f'satellite{FLIGHT_ID}_descriptors.csv'

print('Repo root :', REPO_ROOT)
print('Data root :', DATA_ROOT)
print('Model     :', MODEL_PATH)
print('DB output :', DB_NPZ_PATH)

In [ ]:
from localization.io.dataset import VisLocFlight

flight = VisLocFlight(flight_id=FLIGHT_ID, root=DATA_ROOT)
print('Satellite TIF :', flight.satellite_tif)
print('Metadata CSV  :', flight.metadata_csv)
print('Bounds CSV    :', flight.bounds_csv)
assert flight.satellite_tif.exists(), f'Missing satellite TIF: {flight.satellite_tif}'
assert MODEL_PATH.exists(), f'Missing Mask R-CNN checkpoint: {MODEL_PATH} (it should be in the building-segment download).'

## 5. Load Mask R-CNN (500 × 500 building segmentation backbone)

In [ ]:
import torch
from localization import load_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(
    model_path=str(MODEL_PATH),
    device=device,
    num_classes=2,
    pretrained=False,
).to(device).eval()
print('Device:', device)
print('Model loaded.')

## 6. Build per-patch descriptors

The 9774 × 26762 GeoTIFF tiles into roughly `94 × 263 ≈ 24,722` patches at stride 100. Per patch we typically extract a handful to a few dozen triangle descriptors, depending on building density.

Mask R-CNN inference is batched (`batch_size=4`) so the forward-pass overhead amortizes across patches.

In [ ]:
from localization import build_satellite_descriptors

descriptors_df = build_satellite_descriptors(
    tif_path=str(flight.satellite_tif),
    model=model,
    device=device,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    score_threshold=SCORE_THRESHOLD,
    batch_size=INFERENCE_BATCH_SIZE,
    output_csv=str(DESCRIPTORS_CSV_PATH),
)
print('Total triangle descriptors:', len(descriptors_df))
print('Unique patches            :', descriptors_df['patch_id'].nunique())
descriptors_df.head()

## 7. Build + save `SatelliteDatabase`

In [ ]:
import os
from localization import SatelliteDatabase

db = SatelliteDatabase.from_dataframe(
    descriptors_df,
    parent_tif=os.path.basename(str(flight.satellite_tif)),
    leaf_size=40,
)
db.save(str(DB_NPZ_PATH))
print(db)
print('Saved to:', DB_NPZ_PATH)

## 8. Sanity check: overlay triangle centroids on a downsampled preview

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
with Image.open(flight.satellite_tif) as sat:
    sat_w, sat_h = sat.size
    preview_scale = 2400 / max(sat_w, sat_h)
    preview = sat.resize((int(sat_w * preview_scale), int(sat_h * preview_scale)))

rng = np.random.default_rng(0)
sample_idx = rng.choice(db.size, size=min(2000, db.size), replace=False)
sample_cent = db.centroids[sample_idx] * preview_scale

fig, ax = plt.subplots(figsize=(6, 12))
ax.imshow(preview)
ax.scatter(sample_cent[:, 0], sample_cent[:, 1], s=2, c='yellow', alpha=0.5)
ax.set_title(f'satellite{FLIGHT_ID}.tif ({sat_w}\u00d7{sat_h})\n{db.size:,} triangle centroids (sample shown)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Round-trip sanity check

In [ ]:
db_reloaded = SatelliteDatabase.load(str(DB_NPZ_PATH))
assert db_reloaded.size == db.size
assert np.allclose(db_reloaded.descriptors[:5], db.descriptors[:5])
print(db_reloaded)
print('First 5 descriptors:')
print(db_reloaded.descriptors[:5])

Database is ready. Open [`02_query_uav_localization.ipynb`](02_query_uav_localization.ipynb) to localize a UAV image against this index.